# 🧠 NeuroSync — AI Agent (no API key needed)

Runs **100% free and offline-capable**. No account, no API key.

| Part | What it does |
|---|---|
| **Tools** | calculator, date/time, notes memory, Wikipedia lookup, web page reader |
| **Router** | understands your message and picks the right tool instantly |
| **Local AI brain** (optional) | a small open-source model (Qwen 0.5B) that runs on your own computer for general chat |

**Run:** `Run All` → then chat in the last cell. If the local model can't load, the agent still works with all tools.

## 1. Install (run once)

In [ ]:
# Uncomment on first run
# !pip install -q requests beautifulsoup4 transformers torch

## 2. Imports & settings

In [ ]:
import re, ast, math, json, operator, datetime
from pathlib import Path
from urllib.parse import quote
import requests
from bs4 import BeautifulSoup

USE_LLM = True        # True = smarter chat using a free local model (first run downloads ~1 GB)
                      # False = tools only, starts instantly
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
NOTES_FILE = Path("neurosync_notes.json")
HEADERS = {"User-Agent": "NeuroSync/1.0 (learning project)"}

## 3. Tools

In [ ]:
# ---- Safe calculator ----
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.FloorDiv: operator.floordiv}
_FUNCS = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
          "log": math.log, "log10": math.log10, "abs": abs, "round": round}
_CONSTS = {"pi": math.pi, "e": math.e}

def _eval(n):
    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_eval(n.left), _eval(n.right))
    if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_eval(n.operand))
    if isinstance(n, ast.Name) and n.id in _CONSTS: return _CONSTS[n.id]
    if isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id in _FUNCS:
        return _FUNCS[n.func.id](*[_eval(a) for a in n.args])
    raise ValueError("unsupported")

def calculator(expr):
    try:
        r = _eval(ast.parse(expr, mode="eval").body)
        return str(round(r, 10) if isinstance(r, float) else r)
    except Exception:
        return "Error"

# ---- Time ----
def get_time():
    return datetime.datetime.now().strftime("It is %A, %d %B %Y, %I:%M %p.")

# ---- Notes (long-term memory, saved in a JSON file) ----
def _notes(): return json.loads(NOTES_FILE.read_text()) if NOTES_FILE.exists() else []

def save_note(text):
    notes = _notes()
    notes.append({"id": len(notes) + 1, "text": text,
                  "time": datetime.datetime.now().isoformat(timespec="seconds")})
    NOTES_FILE.write_text(json.dumps(notes, indent=2))
    return f"✅ Saved note #{len(notes)}: {text}"

def search_notes(query=""):
    q = query.lower().strip()
    hits = [n for n in _notes() if q in n["text"].lower()] if q else _notes()
    if not hits: return "I have no matching notes."
    return "\n".join(f"#{n['id']} ({n['time'][:10]}): {n['text']}" for n in hits[-10:])

# ---- Wikipedia (free, no key) ----
def wikipedia(topic):
    try:
        s = requests.get("https://en.wikipedia.org/w/api.php", timeout=10, headers=HEADERS,
                         params={"action": "opensearch", "search": topic, "limit": 1, "format": "json"}).json()
        if not s[1]: return f"I couldn't find '{topic}' on Wikipedia."
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + quote(s[1][0]),
                         timeout=10, headers=HEADERS).json()
        return f"📚 {r.get('title')}: {r.get('extract', 'No summary available.')}"
    except Exception as e:
        return f"Wikipedia lookup failed (check internet): {e}"

# ---- Web page reader + simple summary ----
def read_webpage(url):
    try:
        r = requests.get(url, timeout=15, headers=HEADERS); r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for t in soup(["script", "style", "nav", "footer", "header"]): t.decompose()
        text = re.sub(r"\s+", " ", soup.get_text(" ")).strip()
        sentences = re.split(r"(?<=[.!?])\s+", text)
        good = [s for s in sentences if 40 < len(s) < 300][:4]
        title = soup.title.string.strip() if soup.title and soup.title.string else url
        return f"🌐 {title}\n" + " ".join(good) if good else f"🌐 {title}\n{text[:400]}"
    except Exception as e:
        return f"Couldn't read that page: {e}"

## 4. Local AI brain (free, runs on your computer)

In [ ]:
class LocalBrain:
    """Small open-source chat model. No API key. Falls back gracefully if unavailable."""
    def __init__(self):
        self.ok = False
        if not USE_LLM:
            return
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            print(f"Loading local model {LLM_NAME} (first time downloads ~1 GB)...")
            self.tok = AutoTokenizer.from_pretrained(LLM_NAME)
            self.model = AutoModelForCausalLM.from_pretrained(LLM_NAME)
            self.ok = True
            print("✅ Local AI brain ready.")
        except Exception as e:
            print(f"⚠️ Local model unavailable ({type(e).__name__}). Continuing with tools only.")

    def reply(self, history):
        msgs = [{"role": "system", "content": "You are NeuroSync, a helpful, concise AI assistant."}] + history[-8:]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt")
        out = self.model.generate(**inputs, max_new_tokens=200, do_sample=True,
                                  temperature=0.7, top_p=0.9)
        return self.tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 5. The NeuroSync agent
The router reads your message, chooses a tool, and runs it. If no tool fits, the local AI brain answers.

In [ ]:
HELP = """I\'m NeuroSync. Try:
  • calculate 25*4 + sqrt(81)       • what time is it?
  • remember: submit report Friday   • what do you remember?
  • who is Albert Einstein            • read https://example.com"""

class NeuroSyncAgent:
    def __init__(self):
        self.brain = LocalBrain()
        self.history = []

    def _try_math(self, t):
        expr = re.sub(r"^(calculate|calc|compute|solve|evaluate|what is|what\'s|whats)\s+", "", t)
        expr = expr.rstrip("?= ").replace("^", "**").replace("x", "*") if re.fullmatch(r"[\d\s.+\-*/x^%()]+\??=?", expr) else expr.rstrip("?= ").replace("^", "**")
        if re.search(r"\d", expr) and re.fullmatch(r"[\d\s.+\-*/%(),a-z]+", expr):
            r = calculator(expr)
            if r != "Error": return f"🧮 {r}"
        return None

    def route(self, text):
        t = text.lower().strip()
        if t in {"help", "?", "hi", "hello", "hey"}: return HELP
        if (m := re.match(r"(?:remember|note|save|save note)\s*(?:that|:)?\s*(.+)", text, re.I)):
            return save_note(m.group(1).strip())
        if re.search(r"what do you remember|my notes|show notes|list notes|recall", t):
            return search_notes()
        if (m := re.match(r"(?:search|find) notes?(?: for)?\s+(.+)", t)):
            return search_notes(m.group(1))
        if (m := re.search(r"https?://\S+", text)):
            return read_webpage(m.group(0))
        if re.search(r"\b(time|date|day|today)\b", t) and not re.search(r"\d", t):
            return get_time()
        if (r := self._try_math(t)):
            return r
        if (m := re.match(r"(?:who is|who was|what is|what are|tell me about|wikipedia)\s+(.+)", t)):
            return wikipedia(m.group(1).rstrip("?"))
        return None

    def chat(self, text):
        self.history.append({"role": "user", "content": text})
        answer = self.route(text)
        if answer is None:
            answer = self.brain.reply(self.history) if self.brain.ok else \
                "I\'m not sure about that. Type \'help\' to see what I can do (or set USE_LLM = True for open-ended chat)."
        self.history.append({"role": "assistant", "content": answer})
        return answer

    def reset(self): self.history = []

agent = NeuroSyncAgent()

## 6. Quick demo

In [ ]:
for q in ["calculate (1234 * 56) / 7 + sqrt(144)", "what time is it?",
          "remember: submit the project report on Friday", "what do you remember?",
          "who is Albert Einstein"]:
    print("You:", q); print("NeuroSync:", agent.chat(q), "\n")

## 7. Chat with NeuroSync
`exit` to stop · `reset` to clear conversation · `help` for examples

In [ ]:
def chat_loop():
    print("🧠 NeuroSync ready. Type 'help' for examples, 'exit' to quit.\n")
    while True:
        user = input("You: ").strip()
        if user.lower() in {"exit", "quit"}: print("Goodbye! 👋"); break
        if user.lower() == "reset": agent.reset(); print("Conversation cleared.\n"); continue
        if user: print("\nNeuroSync:", agent.chat(user), "\n")

chat_loop()